### # Clean , Enrich and Validate the raw data
###  # Ensure data quality, eliminate and Optimize performance for downstream analysis

In [0]:
bronze_df=spark.read.table("dev_nyc.bronze.taxi_duartion")
display(bronze_df.limit(5))

In [0]:
from pyspark.sql.functions import col, when, round

cleaned_df = bronze_df.filter(
    (col("Passenger_count") > 0) & (col("Passenger_count") <= 6) &
    (col("trip_distance") > 0.0) & (col("trip_distance") < 100.0) &
    (col("fare_amount") >= 2.50) & (col("fare_amount") <= 100) &
    (col("total_amount") >= col("fare_amount")) &
    (col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
)

deduped_df = cleaned_df.dropDuplicates([
    "VendorID",
    "tpep_pickup_datetime", 
    "PULocationID", 
    "DOLocationID"
])

silver_df = deduped_df \
    .withColumn(
        "trip_duration_minutes", 
        round((col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long")) / 60.0, 2)
    ) \
    .withColumn(
        "tip_percentage",
        when(col("fare_amount") > 0, round((col("tip_amount") / col("fare_amount")) * 100.0, 2))
        .otherwise(0.0)
    ) \
    .withColumn(
        "is_airport_trip",
        when((col("PULocationID").isin(132, 138)) | (col("DOLocationID").isin(132, 138)), True)
        .otherwise(False)
    )

display(silver_df.limit(10))

In [0]:
silver_df.write.format("delta").mode("append").saveAsTable("dev_nyc.silver.taxi_duartion")

In [0]:
display(silver_df)